In [ ]:
# Installazione delle librerie chiave di Hugging Face per QLoRA e Fine-Tuning
!pip install -q -U transformers datasets accelerate peft bitsandbytes trl

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-0.5B-Instruct"

# 1. Configurazione Quantizzazione 4-bit (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                   # 1. Riduce i pesi da 16/32 bit a soli 4 bit
    bnb_4bit_quant_type="nf4",           # 2. Usa il formato NF4 (NormalFloat4), ottimo per le reti neurali
    bnb_4bit_compute_dtype=torch.float16,# 3. Quando la rete fa i calcoli, li fa a 16 bit per non perdere precisione
    bnb_4bit_use_double_quant=True       # 4. "Doppia quantizzazione": comprime anche i metadati della quantizzazione
)

# 2. Caricamento del Tokenizer e del Modello
# il tokenizer trasforma le parole di una frase in numeri
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token # fa padding per completare le frasi più corte con il simbolo di fine frase (eos_token)

# passo tutte le impostazioni al modello preaddestrato
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Modello caricato con successo in 4-bit!")

In [ ]:
from datasets import Dataset
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# 1. Dataset
data = [
    {"t": "Salve, vi scrivo perché il modem spedito lunedì non si accende proprio. Ho provato a cambiare presa ma niente. Voglio il rimborso o disdico tutto subito!", "r": '{"categoria": "Hardware / Guasto", "sentimento": "molto negativo", "priorita": "alta", "azione_richiesta": "Rimborso o sostituzione apparato"}'},
    {"t": "Buongiorno, la velocità della fibra è scesa moltissimo da ieri sera. Riscontro lentezza nello streaming.", "r": '{"categoria": "Connessione / Lentezza", "sentimento": "negativo", "priorita": "media", "azione_richiesta": "Verifica linea e reset segnale"}'},
    {"t": "Vorrei informazioni sul costo del passaggio alla tariffa 2.5 Gbps per i già clienti. Grazie.", "r": '{"categoria": "Commerciale / Info", "sentimento": "neutro", "priorita": "bassa", "azione_richiesta": "Invio prospetto offerte commerciale"}'},
    {"t": "Servizio pessimo!! Siete dei truffatori, mi avete addebitato 50 euro in più in fattura! Pretendo subito una spiegazione o vi denuncio!", "r": '{"categoria": "Fatturazione / Contestazione", "sentimento": "molto negativo", "priorita": "alta", "azione_richiesta": "Verifica amministrativa fattura e storno"}'},
    {"t": "Ho ricevuto il nuovo router stamattina, volevo solo ringraziare l'assistenza per la velocità. Installato e perfettamente funzionante!", "r": '{"categoria": "Feedback / Elogio", "sentimento": "positivo", "priorita": "bassa", "azione_richiesta": "Nessuna azione richiesta"}'},
    {"t": "Non riesco ad accedere all'area clienti dal sito web, mi dà errore 500 dopo il login.", "r": '{"categoria": "Piattaforma / Bug", "sentimento": "negativo", "priorita": "media", "azione_richiesta": "Segnalazione al team IT web"}'},
    {"t": "Mi è arrivata una SIM con numero errato, non corrisponde a quello che avevo richiesto in fase di contratto.", "r": '{"categoria": "Spedizioni / Errore", "sentimento": "negativo", "priorita": "alta", "azione_richiesta": "Invio nuova SIM corretta"}'},
    {"t": "È da due giorni che il telefono fisso non ha segnale, mentre internet funziona benissimo. Potete verificare?", "r": '{"categoria": "Voce / Guasto", "sentimento": "negativo", "priorita": "media", "azione_richiesta": "Controllo centrale telefonica"}'},
    {"t": "Vorrei disdire la mia linea a fine mese per cambio domicilio.", "r": '{"categoria": "Disdetta / Amministrativo", "sentimento": "neutro", "priorita": "media", "azione_richiesta": "Avvio pratica disattivazione"}'},
    {"t": "Complimenti per la gentilezza dell'operatore Marco che mi ha aiutato ieri al telefono!", "r": '{"categoria": "Feedback / Elogio", "sentimento": "molto positivo", "priorita": "bassa", "azione_richiesta": "Nessuna azione richiesta"}'}
]

# 2. Formattazione e Tokenizzazione diretta
texts = []
for item in data:
    prompt = f"""Sei un assistente per l'analisi dei ticket. Estrai le informazioni ed esegui l'output SOLTANTO in formato JSON con le chiavi: categoria, sentimento, priorita, azione_richiesta.

Ticket: "{item['t']}"
"""
    messages = [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": item['r']}
    ]

    # unisce le parti di messages in un'unica stringa (senza convertire in numeri)
    # e aggiunge i tag (apply_chat_template) che servono al modello per capire
    texts.append(tokenizer.apply_chat_template(messages, tokenize=False))

# Tokenizzo tutti i dati
tokenized_inputs = tokenizer(texts,               # dataset in input
                             padding=True,        # aggiunge padding per completare le frasi
                             truncation=True,     # se un testo è troppo lungo lo tronca...
                             max_length=512,      # ... a 512 token
                             return_tensors="pt") # formato di ritorno in tensore pytorch

dataset = Dataset.from_dict({
    "input_ids": tokenized_inputs["input_ids"], # [1543, 8912, 402, 99, 0, 0, 0]
    "attention_mask": tokenized_inputs["attention_mask"] # [1, 1, 1, 1, 0, 0, 0]
})

In [ ]:
# 3. Setup Modello con LoRA
model = prepare_model_for_kbit_training(model) # scongela i pesi solo per i moduli che aggiungo dopo

peft_config = LoraConfig(
    r=16, # dimensione della matrice (il post-it dove memorizzare ad esempio il formato JSON specifico)
    lora_alpha=32, # = 2r per bilanciare i pesi nuovi con quelli precedenti
    target_modules=["q_proj", "v_proj"], # le parti dell'input su cui concentrarsi
    lora_dropout=0.05, # 5% di dropout
    bias="none", # non addestra i vettori del modello originale
    task_type="CAUSAL_LM" # tipo di modello CAUSALE
)
model = get_peft_model(model, peft_config) # assegna la configurazione al modello

# 4. TrainingArguments e Trainer nativo Transformers
training_args = TrainingArguments(
    output_dir="./qwen-ticket-json",     # Cartella temporanea dove salvare i file di log o checkpoint
    per_device_train_batch_size=2,       # Quanti esempi inviare alla GPU alla volta (2 per non saturare la VRAM)
    gradient_accumulation_steps=2,       # Accumula i gradienti per 2 step prima di aggiornare i pesi.
                                         # Simuliamo così un batch size effettivo di 4 (2 * 2), risparmiando RAM
    num_train_epochs=8,                  # Quante volte l'intero dataset (10 esempi) viene fatto vedere al modello
    learning_rate=3e-4,                  # Velocità con cui aggiorna i pesi (0.0003). È il valore ideale per LoRA
    fp16=False,                          # Disattiviamo la gestione automatica fp16 per evitare conflitti con la GPU T4 di Colab
    bf16=False,                          # Disattiviamo bfloat16 non supportato nativamente dall'hardware T4
    logging_steps=1,                     # Stampa la Loss ad ogni singolo step di addestramento
    save_strategy="no",                  # Dice di non salvare checkpoint intermedi su disco ad ogni epoca (risparmia spazio)
    report_to="none"                     # Disattiva l'invio di metriche a piattaforme esterne (come WandB o TensorBoard)
)

trainer = Trainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
)

print("Avvio del Fine-Tuning nativo...")
trainer.train()
print("Addestramento completato!")

In [ ]:
# Ripristino la modalità di generazione
model.config.use_cache = True # riattiva la cache spenta dal metodo prepare_model_for_kbit_training
model.eval() # attiva la modalità inferenza

prompt_test = "Estrai informazioni dal ticket: 'Salve, non riesco più a navigare da ieri'"
messages = [{"role": "user", "content": prompt_test}]

text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to("cuda")

question_length = inputs.input_ids.shape[1] # estraggo la lunghezza della domanda per poi saltarla dopo e mostrare solo la risposta

outputs = model.generate(**inputs, max_new_tokens=100, temperature=0.01)
response = tokenizer.decode(outputs[0][question_length:], skip_special_tokens=True)
print(response)